# Retrieval Inspection

Explore embeddings, FAISS indexing, and retrieval on the handbook chunks before or alongside `src/retrieval/`.

**Goals:**
- Load chunks from `chunks.jsonl`
- Embed chunks and sample queries with sentence-transformers
- Build an in-memory vector index and run searches
- Inspect top-k results, scores, and page refs
- Decide on model, top_k, and indexing choices for `embed`, `vector_store`, `retriever`

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

CHUNKS_JSONL = ROOT / "data" / "processed" / "chunks.jsonl"
print(f"Chunks: {CHUNKS_JSONL}")
print(f"Exists: {CHUNKS_JSONL.exists()}")

Chunks: /Users/srujayreddy/Projects/ca-dmv-rag-system/data/processed/chunks.jsonl
Exists: True


## 2. Load chunks

In [2]:
chunks = []
with open(CHUNKS_JSONL) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        chunks.append(json.loads(line))

texts = [c["text"] for c in chunks]
print(f"Loaded {len(chunks)} chunks")

Loaded 351 chunks


## 3. Embed chunks and queries

In [3]:
from src.retrieval import embed_texts, embed_query

print("Embedding chunks (first run may download the model)...")
embeddings = embed_texts(texts)
print(f"Shape: {embeddings.shape}")

Embedding chunks (first run may download the model)...


/Users/srujayreddy/Projects/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Shape: (351, 384)


In [4]:
# Embed a sample query
query = "What is the blood alcohol limit for DUI?"
q_emb = embed_query(query)
print(f"Query embedding shape: {q_emb.shape}")

Query embedding shape: (384,)


## 4. Build index and search

In [5]:
from src.retrieval.vector_store import VectorStore

store = VectorStore()
store.add(embeddings, chunks)
print(f"Index size: {store.index.ntotal}")

Index size: 351


In [6]:
TOP_K = 5
results = store.search(q_emb, top_k=TOP_K)
print(f"Query: {query}\n")
for i, r in enumerate(results, 1):
    print(f"--- {i} (page {r.get('page')}, score={r.get('score', 0):.4f}) ---")
    print(r["text"][:350] + "..." if len(r["text"]) > 350 else r["text"])
    print()

Query: What is the blood alcohol limit for DUI?

--- 1 (page 79, score=0.7315) ---
73
Blood Alcohol Concentration (BAC) Limits
When you consume alcohol, traces of it enter your bloodstream. Your 
BAC measures how much alcohol is present in your bloodstream.
It is illegal for you to drive if you have a BAC of:
• 0.08% or higher if you are over 21 years old.
• 0.01% or higher if you are under 21 years old.
• 0.01% or higher at any ...

--- 2 (page 80, score=0.5991) ---
 while driving under the influence 
of drugs or alcohol, you may face civil lawsuits.
All DUI convictions remain on your driver’s record for 10 years. If you 
get any other DUIs during that time, the court or DMV may give you an 
additional penalty.

--- 3 (page 81, score=0.5886) ---
ent officer suspects you of consuming alcohol, 
they can require you to take a hand-held breath test, PAS, or another 
chemical test.
• If you are convicted of a DUI with a BAC of 0.01% or higher, DMV may 
revoke your driving privilege for one

## 5. Try more queries

In [7]:
queries = [
    "What is the speed limit on a highway?",
    "How do I renew my driver license?",
    "When must I use headlights?",
]

for q in queries:
    emb = embed_query(q)
    out = store.search(emb, top_k=3)
    print(f"Q: {q}")
    print(f"  Top: page {out[0].get('page')}, score={out[0].get('score', 0):.4f}")
    print()

Q: What is the speed limit on a highway?
  Top: page 73, score=0.6918

Q: How do I renew my driver license?
  Top: page 14, score=0.6654

Q: When must I use headlights?
  Top: page 65, score=0.6337



## 6. Summary & next steps

In [8]:
print("Decisions for src/retrieval:")
print("  - embed: sentence-transformers/all-MiniLM-L6-v2, normalize_embeddings=True")
print("  - vector_store: FAISS IndexFlatIP, metadata in .meta.json")
print("  - retriever: embed_query -> store.search -> top_k chunks with score, page")
print("  - build_index: chunks.jsonl -> embed_texts -> VectorStore.add/save")
print("  - query_demo / Retriever.retrieve: load index, embed query, search")

Decisions for src/retrieval:
  - embed: sentence-transformers/all-MiniLM-L6-v2, normalize_embeddings=True
  - vector_store: FAISS IndexFlatIP, metadata in .meta.json
  - retriever: embed_query -> store.search -> top_k chunks with score, page
  - build_index: chunks.jsonl -> embed_texts -> VectorStore.add/save
  - query_demo / Retriever.retrieve: load index, embed query, search
